# Part 4 — Core API Deep-Dive (Human Example)

*Last updated:* 2026-11-12

This notebook is a **hands-on tutorial** of the public IDTrack API using **human**.

**Learning objectives**
- Create the `idtrack.API` façade and understand what it wraps.
- Convert single identifiers (time travel + optional external outputs).
- Convert batches and summarize outcomes (1→0 / 1→1 / 1→n).
- Request explanation payloads for audit trails.
- Learn advanced knobs (external bridging, ambiguity strategy, assembly awareness).
- Learn introspection helpers (available databases, assemblies, releases, active ranges).

> **Prerequisite:** `03_initialization_graph.ipynb` (Part 3) is recommended so the graph loads from cache.


## 4.1 — The API Facade

`idtrack.API` is the user-facing entry point. It handles:
- organism resolution (species names, common names and taxon IDs of the supported organisms: human, mouse, pig, rhesus and
  cynomolgus macaque)
- building or loading a graph snapshot (the reproducible snapshot boundary)
- conversion helpers like `convert_identifier(...)` and `convert_identifier_multiple(...)`

In this notebook we build (or load) the **human** snapshot, then use it for the rest of the examples.

> **Expected result:** after the setup cell runs, `api.track` exists and conversions become available.


In [1]:
from __future__ import annotations

import os
from pathlib import Path

import idtrack

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
api.configure_logger()

organism, latest_release = api.resolve_organism('human')
SNAPSHOT_RELEASE = 115  # pinned (latest_release is Ensembl's newest release)

api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE)
print('Ready:', organism, 'snapshot', SNAPSHOT_RELEASE)


2026-11-12 15:17:47 INFO:idtrack._verify_organism: Ensembl Rest API query to get the organism names and associated releases.


Ready: homo_sapiens snapshot 115


## 4.2 — Single Identifier Conversion


In [2]:
api.convert_identifier('TP53', to_release=SNAPSHOT_RELEASE)


{'query_id': 'TP53',
 'graph_id': 'TP53',
 'match': 'exact',
 'known_versions': [],
 'from_release': 115,
 'from_release_inferred': True,
 'from_release_given': None,
 'from_release_fallback': False,
 'from_databases': ['HGNC Symbol',
  'NCBI gene (formerly Entrezgene)',
  'UniProtKB Gene Name'],
 'to_release': 115,
 'strategy': 'best',
 'final_database': 'ensembl_gene',
 'target_id': ['ENSG00000141510.20'],
 'last_node': [['ENSG00000141510.20', 'ENSG00000141510.20']],
 'ambiguous': False,
 'tie': False,
 'alternative_locus_genes': ['LRG_321.1'],
 'no_corresponding': False,
 'no_conversion': False,
 'no_target': False,
 'no_target_genes': [],
 'error': None}

If you see `no_corresponding=True`, the input could not be matched to any identifier of the graph. IDTrack already tries the
letter-case and separator variants (`tp53`, `HLA_A` for `HLA-A`) and says in `match` how the identifier matched; `graph_id` is the
identifier of the graph it was read as, and `from_release` the release at which it was read. Without `from_release`, an identifier
is read at the target release if it exists there, otherwise at the latest release at which it exists.

`alternative_locus_genes` lists genes the identifier also leads to that are copies on alternative loci, assembly patches or LRG
records (`LRG_321.1` for TP53): they are reported apart from the answer, which holds the gene of the reference sequence.

### 4.2.1 Example: time travel (convert to an older release)

Why this matters: published datasets often use older releases.


In [3]:
# Choose an older release to demonstrate time travel
older_release = SNAPSHOT_RELEASE - 10
api.convert_identifier('TP53', to_release=older_release)


{'query_id': 'TP53',
 'graph_id': 'TP53',
 'match': 'exact',
 'known_versions': [],
 'from_release': 105,
 'from_release_inferred': True,
 'from_release_given': None,
 'from_release_fallback': False,
 'from_databases': ['HGNC Symbol',
  'NCBI gene (formerly Entrezgene)',
  'UniProtKB Gene Name'],
 'to_release': 105,
 'strategy': 'best',
 'final_database': 'ensembl_gene',
 'target_id': ['ENSG00000141510.18'],
 'last_node': [['ENSG00000141510.18', 'ENSG00000141510.18']],
 'ambiguous': False,
 'tie': False,
 'alternative_locus_genes': ['LRG_321.1'],
 'no_corresponding': False,
 'no_conversion': False,
 'no_target': False,
 'no_target_genes': [],
 'error': None}

### 4.2.2 Convert into an external database (HGNC)

To convert into a specific external database, pass `final_database=...`.
Database names are the same names you see in your external YAML.


In [4]:
api.convert_identifier('ENSG00000141510', to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol')


{'query_id': 'ENSG00000141510',
 'graph_id': 'ENSG00000141510',
 'match': 'exact',
 'known_versions': [],
 'from_release': 115,
 'from_release_inferred': True,
 'from_release_given': None,
 'from_release_fallback': False,
 'from_databases': ['base_ensembl_gene'],
 'to_release': 115,
 'strategy': 'best',
 'final_database': 'HGNC Symbol',
 'target_id': ['TP53'],
 'last_node': [['ENSG00000141510.20', 'TP53']],
 'ambiguous': False,
 'tie': False,
 'alternative_locus_genes': [],
 'no_corresponding': False,
 'no_conversion': False,
 'no_target': False,
 'no_target_genes': [],
 'error': None}

### 4.2.3 How do I know which external databases are available?

Use the graph itself to list databases currently represented.


In [5]:
g = api.track.graph
sorted(g.available_external_databases)[:50]


['CCDS',
 'Clone_based_ensembl_gene',
 'Clone_based_vega_gene',
 'EntrezGene',
 'HGNC Symbol',
 'Havana gene',
 'Havana transcript',
 'Havana translation',
 'NCBI gene',
 'NCBI gene (formerly Entrezgene)',
 'RFAM',
 'RefSeq_mRNA',
 'RefSeq_mRNA_predicted',
 'RefSeq_ncRNA',
 'RefSeq_ncRNA_predicted',
 'RefSeq_peptide',
 'RefSeq_peptide_predicted',
 'UniProtKB Gene Name',
 'Uniprot/SPTREMBL',
 'Uniprot/SWISSPROT',
 'Vega gene',
 'Vega_gene',
 'synonym_id::EntrezGene',
 'synonym_id::HGNC Symbol',
 'synonym_id::NCBI gene',
 'synonym_id::NCBI gene (formerly Entrezgene)',
 'synonym_id::UniProtKB Gene Name']

### 4.2.4 Understanding the result dictionary

Key fields (the docstring of `API.convert_identifier` lists every field):
- `query_id`: exactly what you typed
- `graph_id`: the identifier of the graph it was read as; `match`: how it matched (`exact`, `case`, `separator`, `version`,
  `synonym:...`, `version_newer_than_graph`)
- `from_release`: the release it was read at; `from_release_inferred`: chosen by IDTrack (no `from_release` given);
  `from_release_fallback`: the `from_release` you gave does not hold the identifier, so it was read where the graph holds it
- `target_id`: list of outputs (can be 0, 1, or many), in a fixed order; `last_node`: `[gene, target]` pairs
- `ambiguous`: more than one target; `tie`: the best answer is shared by several genes or targets equal in all evidence
- `alternative_locus_genes`: copies on alternative loci and patches reported apart from the answer
- `no_corresponding`: input didn’t match any identifier of the graph (`known_versions` lists the versions that exist when you gave
  an Ensembl id with a version the graph does not hold)
- `no_conversion`: input matched, but no gene at the target release descends from it
- `no_target`: genes were reached, but none has an id in the requested database (the targets are then their Ensembl ids);
  `no_target_genes`: genes without such an id when others have one

Important: `target_id` is a list because ambiguity is real and common. Every result is plain data (`json.dumps` works on it).

### 4.2.5 Versions, spelling variants and releases

A few identifiers that show how IDTrack reads what you give it:
- an Ensembl id with a version that never existed is not answered with another version's gene (`known_versions` names the ones that
  exist), except a version higher than all the graph holds of an id still alive at the snapshot, which may come from a newer
  release and is read as the latest version;
- a `from_release` at which the graph does not hold the identifier is set aside and reported (`from_release_fallback`), while a
  release that is not a release of the graph at all is an error that names the valid ones.

In [6]:
examples = [
    ('ENSG00000141510.5', {}),                      # an old version: read where it exists (release 54), followed to 115
    ('ENSG00000141510.3', {}),                      # a version the graph does not hold (its first is .4): nothing
    ('ENSG00000141510.25', {}),                     # newer than the graph's latest version: read as the latest version
    ('hla_a', {}),                                  # letter case and separator variant of HLA-A
    ('NM_000546.9', {}),                            # a RefSeq accession with a version the graph does not hold
    ('TP53', {'from_release': 50, 'to_release': 50}),  # read and converted at an NCBI36 release
    ('ENSG00000141510.20', {'from_release': 100}),  # this version did not exist at 100: read where it exists, flagged
    ('ENSG00000141510', {'from_release': 40}),      # not a release of the graph: a ValueError names the valid ones
]
for query, kwargs in examples:
    try:
        r = api.convert_identifier(query, **kwargs)
    except ValueError as error:
        print(f'{query!r} {kwargs} -> ValueError: {error}')
        continue
    versions = r['known_versions']
    known = f' known_versions: {len(versions)}, {versions[0]} to {versions[-1]}' if versions else ''
    print(f"{query!r} {kwargs} -> {r['target_id']} | graph_id={r['graph_id']!r} match={r['match']!r} "
          f"from_release={r['from_release']} fallback={r['from_release_fallback']} no_corresponding={r['no_corresponding']}{known}")

'ENSG00000141510.5' {} -> ['ENSG00000141510.20'] | graph_id='ENSG00000141510.5' match='exact' from_release=54 fallback=False no_corresponding=False


'ENSG00000141510.3' {} -> [] | graph_id=None match=None from_release=None fallback=False no_corresponding=True known_versions: 17, ENSG00000141510.4 to ENSG00000141510.20
'ENSG00000141510.25' {} -> ['ENSG00000141510.20'] | graph_id='ENSG00000141510.20' match='version_newer_than_graph' from_release=115 fallback=False no_corresponding=False
'hla_a' {} -> ['ENSG00000206503.16'] | graph_id='HLA-A' match='separator' from_release=115 fallback=False no_corresponding=False
'NM_000546.9' {} -> ['ENSG00000141510.20'] | graph_id='NM_000546' match='version' from_release=115 fallback=False no_corresponding=False
'TP53' {'from_release': 50, 'to_release': 50} -> ['ENSG00000141510.5'] | graph_id='TP53' match='exact' from_release=50 fallback=False no_corresponding=False
'ENSG00000141510.20' {'from_release': 100} -> ['ENSG00000141510.20'] | graph_id='ENSG00000141510.20' match='exact' from_release=115 fallback=True no_corresponding=False
'ENSG00000141510' {'from_release': 40} -> ValueError: `from_release

### 4.2.6 Ambiguity control: strategy='best' vs strategy='all'

- `strategy='best'` (default): returns the best answer: one target, or, when several genes or targets are equal in all evidence,
  all of them in ascending order of gene and target id with `tie=True`
- `strategy='all'`: returns *all* targets IDTrack found, best first

Use `'all'` when you are doing QC or want to inspect ambiguous mappings.


In [7]:
api.convert_identifier('TP53', to_release=SNAPSHOT_RELEASE, strategy='all')


{'query_id': 'TP53',
 'graph_id': 'TP53',
 'match': 'exact',
 'known_versions': [],
 'from_release': 115,
 'from_release_inferred': True,
 'from_release_given': None,
 'from_release_fallback': False,
 'from_databases': ['HGNC Symbol',
  'NCBI gene (formerly Entrezgene)',
  'UniProtKB Gene Name'],
 'to_release': 115,
 'strategy': 'all',
 'final_database': 'ensembl_gene',
 'target_id': ['ENSG00000141510.20'],
 'last_node': [['ENSG00000141510.20', 'ENSG00000141510.20']],
 'ambiguous': False,
 'tie': False,
 'alternative_locus_genes': ['LRG_321.1'],
 'no_corresponding': False,
 'no_conversion': False,
 'no_target': False,
 'no_target_genes': [],
 'error': None}

## 4.3 — Batch Conversion

Most workflows start from a list of identifiers (genes in a count matrix, markers, hits, etc.).
IDTrack provides two helpers:
- `convert_identifier_multiple(...)`
- `classify_multiple_conversion(...)` to summarize outcomes


In [8]:
genes = ['TP53', 'BRCA1', 'BRCA2', 'BRAF', 'KRAS', 'NOT_A_REAL_GENE']
results = api.convert_identifier_multiple(genes, to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol')
results[:2]  # show first two


  0%|                                                                         | 0/6 [00:00<?, ?it/s]

100%|████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 155.68it/s]

[{'query_id': 'TP53',
  'graph_id': 'TP53',
  'match': 'exact',
  'known_versions': [],
  'from_release': 115,
  'from_release_inferred': True,
  'from_release_given': None,
  'from_release_fallback': False,
  'from_databases': ['HGNC Symbol',
   'NCBI gene (formerly Entrezgene)',
   'UniProtKB Gene Name'],
  'to_release': 115,
  'strategy': 'best',
  'final_database': 'HGNC Symbol',
  'target_id': ['TP53'],
  'last_node': [['ENSG00000141510.20', 'TP53']],
  'ambiguous': False,
  'tie': False,
  'alternative_locus_genes': ['LRG_321.1'],
  'no_corresponding': False,
  'no_conversion': False,
  'no_target': False,
  'no_target_genes': [],
  'error': None},
 {'query_id': 'BRCA1',
  'graph_id': 'BRCA1',
  'match': 'exact',
  'known_versions': [],
  'from_release': 115,
  'from_release_inferred': True,
  'from_release_given': None,
  'from_release_fallback': False,
  'from_databases': ['HGNC Symbol',
   'NCBI gene (formerly Entrezgene)',
   'UniProtKB Gene Name'],
  'to_release': 115,
  'st

In [9]:
summary = api.classify_multiple_conversion(results)
# Each bin is a list of per-gene dictionaries
{k: len(v) for k, v in summary.items()}


{'changed_only_1_to_n': 0,
 'changed_only_1_to_1': 0,
 'alternative_target_1_to_1': 0,
 'alternative_target_1_to_n': 0,
 'matching_1_to_0': 1,
 'matching_1_to_1': 5,
 'matching_1_to_n': 0,
 'input_identifiers': 6}

If you want a human-readable report, you can print the summary bins:


In [10]:
api.print_binned_conversion(summary)


2026-11-12 15:17:52 INFO:idtrack._api: 
IDTrack conversion summary:
  Total processed: 6
  1→0: 1 (16.7%)
  1→1: 5 (83.3%)
    Changed only: 0 (0.0%)
    Alternative targets: 0 (0.0%)
    Rest: 5 (100.0%)
  1→n: 0 (0.0%)
    Changed only: 0 (0.0%)
    Alternative targets: 0 (0.0%)
  Diagnostics:
    no_corresponding: 1
    no_conversion:   0
    no_target:       0
    errors:          0


## 4.4 — Explainability & Auditability

When you set `explain=True`, the result includes a `the_path` field: one entry per `[gene, target]` pair of `last_node`, with the gene
the identifier was read as (`entry_gene`), the number of bridges through external identifiers the search needed (`external_jump`; 0 is
Ensembl's history alone), the releases between leaving one gene and entering the next over those bridges (`external_gap`), the product
of Ensembl's mapping scores along the history (`score`), and the edges walked (`edges`): `[source, target, key]` for a step of
Ensembl's history in walking order, `[source, target, 0, release]` for a cross-reference followed at that release.

In [11]:
explained = api.convert_identifier('TP53', to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol', explain=True)
list(explained.keys())


['query_id',
 'graph_id',
 'match',
 'known_versions',
 'from_release',
 'from_release_inferred',
 'from_release_given',
 'from_release_fallback',
 'from_databases',
 'to_release',
 'strategy',
 'final_database',
 'target_id',
 'last_node',
 'ambiguous',
 'tie',
 'alternative_locus_genes',
 'no_corresponding',
 'no_conversion',
 'no_target',
 'no_target_genes',
 'error',
 'the_path']

In [12]:
# One entry per [gene, target] pair of `last_node`, in the same order
explained['the_path'][0]


{'ensembl_gene': 'ENSG00000141510.20',
 'target_id': 'TP53',
 'entry_gene': 'ENSG00000141510.20',
 'external_jump': 0,
 'external_gap': 0,
 'score': 1.0,
 'edges': [['TP53', 'ENSG00000141510.20', 0, 115],
  ['ENSG00000141510.20', 'TP53', 0, 115]]}

`the_path` is intentionally detailed. For most users, the summary flags and `target_id` are enough.


## 4.5 — Advanced Conversion Options

`api.convert_identifier(...)` is a convenience wrapper around `api.track.convert(...)`.

Use the high-level API most of the time.
But if you need full control (whether external bridging is allowed, the search budget, deeper path diagnostics), you can call
`Track.convert` directly. It is strict where the API is forgiving: a `from_release` at which the identifier does not exist raises
`idtrack.InactiveAtReleaseError` (naming the releases where it exists), and an unknown version raises `idtrack.UnknownVersionError`.

### 4.5.1 Best vs all (selection strategy)

- `strategy='best'` returns the best answer: one target, or the targets that are equal in all evidence, as a tie (`tie=True`), in a
  fixed order.
- `strategy='all'` returns all targets reached, best first (useful for ambiguity-aware pipelines).

### 4.5.2 Controlling external bridging

The search follows Ensembl's history first. Only when that leads to no gene at the target release does it pass through an external
identifier or a transcript that two genes carried (a bridge): at most two bridges, the fewest possible, in the order of time, never
through clone-based names or hyper-connected identifiers. `external_jump` in the result says how many bridges were used. Power users
can switch bridging off with `go_external=False` on `Track.convert`. The search is bounded: each state is visited once, and a call
that needs more than its budget (`Track.search_budget_states`, `Track.search_budget_seconds`) raises
`idtrack.SearchBudgetExceededError`.

### 4.5.3 Hyperconnected nodes

Some external identifiers are carried by many genes at the same time (RFAM families, `U6`, `5S_rRNA`, snoRNA names): more than
`TheGraph.genes_at_once_threshold` (5) genes in one release. They name a kind of gene, not a gene, so they are never used as bridges
and are not counted as annotation of a gene. They can still be queried and returned.

### 4.5.4 Assembly-aware conversions

In IDTrack, genome assemblies are part of the graph. This is crucial when you integrate datasets that were annotated with different
references (for example, a GRCh37-based GTF and a GRCh38-based GTF).

A conversion uses, at every release, the assembly Ensembl maintained at that release (human: NCBI36 up to release 54, GRCh37 up to
75, GRCh38 from 76). The parallel GRCh37 of later releases is not walked; its ids are in the graph as ids to convert from. The final
conversion into a database is made on the assembly of the target release, so a conversion to release 70 returns the HGNC symbols of
GRCh37 at release 70.

Practical consequences:
- You can feed identifiers originating from older builds and harmonize them into one target space (a release of your snapshot).
- External databases are read for the assembly they belong to; an id is returned as its database wrote it at the target release.

Example (advanced):
```python
api.track.convert(
    from_id='TP53',
    from_release=None,
    to_release=SNAPSHOT_RELEASE,
    final_database='HGNC Symbol',
    go_external=True,
    return_path=True,
    from_database='HGNC Symbol',  # read the identifier as an id of this database only
)
```

In [13]:
# Advanced demo: inspect hyperconnected nodes and compare `go_external` behavior.
# Safe: does not modify your cache; it only runs conversions.

# 1) Hyperconnected nodes: external ids carried by more than `genes_at_once_threshold` genes in one release
g = api.track.graph
hc = g.hyperconnective_nodes
print('Threshold (genes at once):', g.genes_at_once_threshold)
print('Hyperconnected external nodes:', len(hc))
top = sorted(hc.items(), key=lambda kv: (-kv[1], kv[0]))[:10]
print('Top 10 by the number of genes they are linked to in one release:')
for node, genes in top:
    print(' ', genes, '-', node)

# 2) External bridging toggle (often matters when backbone history is disconnected)
# For many well-behaved genes, both calls will succeed; the point is the *option* exists.
res_no_external = api.track.convert(
    from_id='TP53',
    from_release=None,
    to_release=SNAPSHOT_RELEASE,
    final_database=None,
    go_external=False,
    prioritize_to_one_filter=True,
    return_path=False,
)
res_with_external = api.track.convert(
    from_id='TP53',
    from_release=None,
    to_release=SNAPSHOT_RELEASE,
    final_database=None,
    go_external=True,
    prioritize_to_one_filter=True,
    return_path=False,
)

for label, res in (('go_external=False', res_no_external), ('go_external=True ', res_with_external)):
    print(label, '->', {gene: value['external_jump'] for gene, value in res.items()} if res else None)


Threshold (genes at once): 5
Hyperconnected external nodes: 10825
Top 10 by the number of genes they are linked to in one release:
  1562 - RF00026
  1562 - U6
  1266 - Metazoa_SRP
  1266 - RF00017
  1013 - RF00019
  1013 - Y_RNA
  615 - 5S_rRNA
  615 - RF00001
  480 - RF01210
  480 - snoU13
go_external=False -> {'ENSG00000141510.20': 0}
go_external=True  -> {'ENSG00000141510.20': 0}


## 4.6 — Introspection & Discovery

These helpers answer practical questions like:
- *Which external databases are available in my current graph?*
- *Which genome assemblies are represented?*
- *What Ensembl release range does my snapshot cover?*
- *When was a given identifier active across releases?*

The next cell demonstrates the most useful introspection calls.


In [14]:
# Introspection demo

print('Assemblies in this graph:', sorted(api.list_genome_assemblies()))

ext_dbs = sorted(api.list_external_databases())
print('External DBs enabled (count):', len(ext_dbs))
print('External DBs (first 25):', ext_dbs[:25])

forms = api.external_database_forms()
print()
print('External DB connection forms (sample):')
for name in ext_dbs[:10]:
    print(' ', name, '→', forms.get(name))

rels = api.list_ensembl_releases()
print()
print('Ensembl releases in snapshot window:', (min(rels), max(rels)) if rels else None)
print('Releases of each assembly:', {a: (min(r), max(r)) for a, r in sorted(api.track.graph.assembly_releases.items())})

# Active ranges: when was an ID "alive" across releases?
# (Useful for provenance documentation.)
g = api.track.graph
example_gene = 'ENSG00000141510.20'  # TP53 at the snapshot release
print()
print('Versions of ENSG00000141510:', g.versions_of('ENSG00000141510'))
print('Active ranges of', example_gene, '(the assembly Ensembl maintained at each release):', g.active_ranges(example_gene))
print('Active ranges of', example_gene, '(any assembly):', g.get_active_ranges_of_id_ensembl_all_inclusive(example_gene))


2026-11-12 15:17:52 INFO:idtrack._the_graph: Graph-wide table being calculated: assembly_releases


Assemblies in this graph: [36, 37, 38]
External DBs enabled (count): 27
External DBs (first 25): ['CCDS', 'Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'EntrezGene', 'HGNC Symbol', 'Havana gene', 'Havana transcript', 'Havana translation', 'NCBI gene', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RefSeq_mRNA', 'RefSeq_mRNA_predicted', 'RefSeq_ncRNA', 'RefSeq_ncRNA_predicted', 'RefSeq_peptide', 'RefSeq_peptide_predicted', 'UniProtKB Gene Name', 'Uniprot/SPTREMBL', 'Uniprot/SWISSPROT', 'Vega gene', 'Vega_gene', 'synonym_id::EntrezGene', 'synonym_id::HGNC Symbol', 'synonym_id::NCBI gene']

External DB connection forms (sample):
  CCDS → transcript
  Clone_based_ensembl_gene → gene
  Clone_based_vega_gene → gene
  EntrezGene → gene
  HGNC Symbol → gene
  Havana gene → gene
  Havana transcript → transcript
  Havana translation → translation
  NCBI gene → gene
  NCBI gene (formerly Entrezgene) → gene

Ensembl releases in snapshot window: (48, 115)


Releases of each assembly: {36: (48, 54), 37: (55, 115), 38: (76, 115)}

Versions of ENSG00000141510: ('ENSG00000141510.4', 'ENSG00000141510.5', 'ENSG00000141510.6', 'ENSG00000141510.7', 'ENSG00000141510.8', 'ENSG00000141510.9', 'ENSG00000141510.10', 'ENSG00000141510.11', 'ENSG00000141510.12', 'ENSG00000141510.13', 'ENSG00000141510.14', 'ENSG00000141510.15', 'ENSG00000141510.16', 'ENSG00000141510.17', 'ENSG00000141510.18', 'ENSG00000141510.19', 'ENSG00000141510.20')
Active ranges of ENSG00000141510.20 (the assembly Ensembl maintained at each release): [[115, inf]]
Active ranges of ENSG00000141510.20 (any assembly): [[115, 115]]


## 4.7 — Practical advice (the kind that saves you a week)

1. Always record your **snapshot boundary** (release) in your analysis notes.
2. If you share results, share the **external YAML** too.
3. When mapping is ambiguous, do not hide it — decide how your pipeline should handle 1→n mappings.
4. For scRNA-seq harmonization, prefer stable namespaces (Ensembl IDs) before switching to symbols.

> **Tip:** If you need troubleshooting checklists and diagnostics helpers, see `07_advanced_topics.ipynb` (Part 7.3).
